<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 2 (Martes) — Clasificación 1 — Árboles de decisión y bosques aleatorios**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🌳 Árboles de decisión y bosques aleatorios: teoría y construcción desde cero

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Entender qué es un árbol de decisión, cómo mide la pureza (Gini, entropía) y qué es un bosque aleatorio. |
| 🔍 Comprensión de los datos | Cargamos los datos del Titanic (`Age`, `Fare` → `Survived`) y de abandono de clientes (`telecom_churn.csv`). |
| 🧹 Preparación de los datos | Seleccionamos variables, eliminamos faltantes y fijamos hiperparámetros (`max_depth`, `min_samples_split`). |
| 🤖 Modelado | Programamos `Node` y `RandomForestClassifier` desde cero y visualizamos el árbol con Graphviz. |
| 📊 Evaluación | Evaluamos con curva ROC/AUC, matriz de confusión, exactitud, sensibilidad y especificidad. |
| 🚀 Despliegue y conclusiones | Interpretamos las métricas y cerramos con una práctica sobre sobreajuste con `scikit-learn`. |

> 🎯 **Analogía:** un árbol de decisión es el juego de las "20 preguntas": cada pregunta (nodo) separa los casos lo mejor posible; un bosque es consultar a muchos jugadores distintos y quedarse con el voto de la mayoría.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 👉 **2** | Martes | **Clasificación 1: árboles de decisión, bosques aleatorios y KNN** |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('train.csv', 'dia_3'), ('telecom_churn.csv', 'dia_3')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Entender qué es un árbol de decisión, cómo mide la pureza (Gini, entropía) y qué es un bosque aleatorio.

📚 Lectura recomendada: [3] [4]

# **Árbol de decisión**

Un árbol de decisión es una especie de mapa en que se muestra cada una de las opciones de decisión posibles y sus resultados. Este es tremendamente útil para aquellas personas que tienen que tomar decisiones en un negocio, ya que te permite comparar diferentes decisiones y acciones según sus costos, probabilidades y beneficios



![Árbol de decisión: nodo interno, ramas y hojas](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_17.png)
*Figura: Árbol de decisión: nodo interno, ramas y hojas. [10]*

**Ejemplo grafico**

Un árbol de decisiones es exactamente lo que implica su nombre. Una estructura en forma de árbol que permite modelar decisiones y sus consecuencias.

Visualicemos el árbol de decisiones que de un juego para comprender mejor cómo se guio la clasificación de un personaje en el que estaba pensando "Iron Man"

![Árbol de decisión entrenado con Iris (sklearn.plot_tree)](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_21.png)
*Figura: Árbol de decisión entrenado con Iris (sklearn.plot_tree). [10]*

El árbol de decisión consta de nodos y aristas que apuntan desde los nodos principales a los nodos secundarios. En nuestro caso, cada nodo contenía una pregunta como "¿Es una persona?" mientras que los ramas contenían las respectivas respuestas: "Sí" o "No". En nuestro caso, las etiquetas eran binarias, lo que significa que cada pregunta puede responderse de dos formas: "Sí" o "No".

Mientras estábamos jugando, se camina por el árbol comenzando con el nodo raíz en la parte superior, siguiendo las ramas y los nodos hasta que finalmente se llego al nodo que no tenía una rama saliente: la clasificacion.

### **Tipos de árboles de decisión**

El **árbol de clasificación** es un árbol donde la predicción es categórica. El árbol que hemos construido arriba es un árbol de clasificación ya que su salida siempre dará un resultado de una categoría como "Superhéroes" o más específicamente **"Iron Man"**.





El **árbol de regresión** es un árbol en el que las salidas pueden ser continuas. Un ejemplo de valores continuos son los números reales .





## **CART**

(Por sus siglas en Inglés Classification and Regression Trees o árboles de clasificación y regresión), es el algoritmo **más popular** para realizar arboles de decisión en aprendizaje maquina






Cual es el punto optimo de la partición

![Regiones de decisión de un árbol sobre el conjunto Iris](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_20.png)
*Figura: Regiones de decisión de un árbol sobre el conjunto Iris. [10]*

**1. Impureza de Gini**


Gini es una medida de impureza. Cuando Gini vale 0, significa que ese nodo es totalmente puro. La impureza se refiere a cómo de mezcladas están las clases en cada nodo







El cálculo de la Impureza de Gini de lo anterior sería el siguiente:






La fórmula es la siguiente:

$$ $$

$$
G=\sum_{-} i=1^{C} p_{i} \times\left(1-p_{i}\right)
$$

donde $C$ son las clases y $p_{i}$ es la probabilidad de elegir un elemento de clase $i$ al azar.

![Entropía, Gini y error de clasificación según p(i=1)](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_19.png)
*Figura: Entropía, Gini y error de clasificación según p(i=1). [10]*

Ejemplo de **Impureza de Gini**

Supongamos que hemos recopilado algunos datos meteorológicos durante los últimos años y queremos usarlos para determinar si deberíamos llevar un paraguas con nosotros dadas las condiciones climáticas actuales que observamos.

$$ $$
Entre otras cosas, hemos recopilado 14 puntos de datos sobre el viento y la temperatura :

$$ $$

\begin{array}{|c|c|c|c|}
\hline \text { Viento } & \text { Si } & \text { No } & \text { Total } \\
\hline \text { Debil } & 4 & 3 & 7 \\
\hline \text { Fuerte } & 6 & 1 & 7 \\
\hline & & & 14 \\
\hline
\end{array}


$$ $$


\begin{array}{|c|c|c|c|}
\hline \text { Temperatura } & \text { Si } & \text { No } & \text { Total } \\
\hline \text { Alto } & 3 & 2 & 5 \\
\hline \text { Bajo } & 7 & 2 & 9 \\
\hline & & & 14 \\
\hline
\end{array}

$$ $$

Como se puede ver, hemos categorizado el viento en "débil" y "fuerte" y la temperatura en "alta" y "baja". Durante un día donde el viento era "fuerte" llovió 6 sobre 7 veces. Al mismo tiempo llovió 7 veces de 9 momentos en los que la temperatura era "baja".

$$ $$
Nuestro objetivo es determinar cuál de estas observaciones nos brinda más certeza de si va a llover y, por lo tanto, si debemos llevar un paraguas. Podemos averiguarlo aplicando la fórmula de **Gini Impureza**.
$$ $$
Comencemos con el cálculo de la impureza de Gini para el viento.
$$ $$
Estamos tratando con 2 casos separados aquí, ya que puede haber viento "débil" o "fuerte". Para cada caso que tenemos 2 diferentes clases $(C)$ que podemos usar para etiquetar nuestra observación: "Sí" y "No". Esto significa que tenemos que hacer 2 cálculos de impurezas de Gini (uno para cada caso).
Conectar los números da como resultado lo siguiente:

$$
\begin{aligned}
\\
&\text { Gini }(\text { Viento }=\text { Debil })=\frac{4}{7} \times\left(1-\frac{4}{7}\right)+\frac{3}{7} \times\left(1-\frac{3}{7}\right)=\frac{24}{49} \approx 0.49 \\
\\
&\text { Gini }(\text { Viento }=\text { Fuerte })=\frac{6}{7} \times\left(1-\frac{6}{7}\right)+\frac{1}{7} \times\left(1-\frac{1}{7}\right)=\frac{12}{49} \approx 0.24
\end{aligned}
\\
\\
$$

$$ $$


Como tenemos 2 resultados (uno para cada caso), ahora necesitamos calcular la suma ponderada para obtener el resultado general de nuestra observación del viento :
$$ $$


$$
\text { Gini }(\text { Viento })=\frac{7}{14} \times 0.49+\frac{7}{14} \times 0.24 \approx 0.37
$$

$$ $$

A continuación, hacemos el cálculo de la impureza de Gini para la temperatura . Estamos siguiendo exactamente los mismos pasos que hicimos con nuestro cálculo de viento anterior. También estamos tratando con 2 casos diferentes ("Alto" y "Bajo"), así como 2 diferentes clases $(C)$ ("Si y no"). Aquí está el cálculo:


$$ $$


$$
\begin{gathered}
\operatorname{Gini}(\text { Temperatura }=\text { Bajo })=\frac{3}{5} \times\left(1-\frac{3}{5}\right)+\frac{2}{5} \times\left(1-\frac{2}{5}\right)=\frac{12}{25} \approx 0.48 \\
\\
\operatorname{Gini}(\text { Temperatura }=\text { Alto })=\frac{7}{9} \times\left(1-\frac{7}{9}\right)+\frac{2}{9} \times\left(1-\frac{2}{9}\right)=\frac{28}{81} \approx 0.35 \\
\\
\end{gathered}
$$


\begin{equation}
\text { Gini }(\text { Temperatura })=\frac{5}{14}  \times 0.48+\frac{9}{14} \times 0.35 \approx 0.4
\end{equation}



**Ejemplo de pureza**

*   0 máxima pureza
*   0.278 pureza media
*   0.444 pureza media baja
*   0.5 pureza baja
*   1 impureza total

**¿Cuál es el análisis es este árbol de decisión?**

![Dos divisiones posibles del mismo nodo (40, 40)](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_18.png)
*Figura: Dos divisiones posibles del mismo nodo (40, 40). [10]*

**2. Entropía / ganancia de la información**

La entropía es la cantidad de desorden de la información, o la cantidad de aleatoriedad en los datos. La entropía en un nodo depende de la cantidad de datos aleatorios que se encuentran en ese nodo y se debe calcular para cada nodo.

La ganancia de información calcula la diferencia entre la entropía antes de la división y la entropía promedio después de la división del conjunto de datos en función de los valores de atributo dados


La entropía y la información son opuestos



$$ $$

**Formula Entropia**

Sea $\ p_{i}$ la fraccion de ejemplos que pertenecen a la clase $\ {n}$ el numero de clases. En el calculo de la entropıa se asume que
log2
(0) = 0.

$$ $$

\begin{equation}
\text { Entropy }=-\sum_{i=1}^{n} p_{i} .\log _{2}  p_{i}
\end{equation}

$$$$

**Formula ganancia de la información**

$$ $$
$$
\text { Gain }=E_{\text {parent }}-E_{\text {children }}
$$

$$
\begin{gathered}
\text { Gain }=1-0.39 \\
=0.61
\end{gathered}
$$

**Ejemplo**



$$$$
La probabilidad de ganar con el canasta 1 es 1
$$$$

$$$$
La probabilidad de ganar con canasta 2 es 0.105
$$$$

$$$$
La probabilidad de ganar con el canasta 3 es 0.0625

$$$$



---


$$$$
Como se resume en la siguiente tabla:



Ok, ahora tenemos una medida que nos da diferentes valores para las tres canastas. La probabilidad de ganar en este juego nos da:
$$$$

*   1.0 para la canasta 1
*   0.105 para la canasta 2
*   0.0625 para la canasta 3

$$$$
Para construir la fórmula de entropía, queremos lo contrario, alguna medida que nos dé un número bajo para la canasta 1, un número medio para la canasta 2 y un número alto para la canasta 3.

$$ $$

Alplicar la formula general de entropia
$$ $$
Ahora, como paso final, tomamos el promedio, con el fin de normalizar. ¡Y eso es todo, esa es la entropía!
$$ $$

Para la canasta 2, es $0.811$ :

$$ $$

$$
\frac{1}{4}\left(-\log _{2}(0.75)-\log _{2}(0.75)-\log _{2}(0.75)-\log _{2}(0.25)\right)=0.81125
$$

$$ $$
Entropía para la cansta 1, es $ 0 $
$$ $$
Si calculamos la entropía para la canasta 1 (4 bolas rojas), obtenemos:
$$ $$
$$
\frac{1}{4}\left(-\log _{2}(1)-\log _{2}(1)-\log _{2}(1)-\log _{2}(1)\right)=0
$$


$$ $$
Entropía para la cansta 3, es $ 1 $  (2 bolas rojas, 2 bolas azules), obtenemos:
$$ $$
$$
\frac{1}{4}\left(-\log _{2} 0.5-\log _{2} 0.5-\log _{2} 0.5-\log _{2} 0.5\right)=1
$$
$$ $$

Entonces tenemos nuestra fórmula para la entropía, el logaritmo negativo de la probabilidad de ganar en nuestro juego. Tenga en cuenta que es **bajo** para la canasta 1, **alto** para la canasta 3 y **medio** la canasta 2.

$$ $$
Como se resume en la siguiente tabla:

![Árbol con criterio de entropía y valores por nodo](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_21_1.png)
*Figura: Árbol con criterio de entropía y valores por nodo. [10]*

---
## 🤖 Fase 4 · Modelado — árbol de decisión desde cero

Programamos `Node` y `RandomForestClassifier` desde cero y visualizamos el árbol con Graphviz.

📚 Lectura recomendada: [5] [9]

## **Modelado desde 0 arbol de decisión**


In [ ]:
!apt-get -qq install -y graphviz
%pip install -q graphviz pydotplus

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import graphviz
from graphviz import Digraph
from IPython.display import Image

In [ ]:
class Node:
    """
    Clase para crear los nodos para un árbol de decisiones
    """
    def __init__(
        self,
        Y: list,
        X: pd.DataFrame,
        min_samples_split=None,
        max_depth=None,
        depth=None,
        node_type=None,
        rule=None
    ):
        # Guardar los datos en el nodo
        self.Y = Y
        self.X = X

        # Guardar los hiperparámetros
        self.min_samples_split = min_samples_split if min_samples_split else 20
        self.max_depth = max_depth if max_depth else 5

        # Profundidad actual predeterminada del nodo
        self.depth = depth if depth else 0

        # Extrayendo todas las características
        self.features = list(self.X.columns)

        # Tipo de nodo
        self.node_type = node_type if node_type else 'root'

        # Regla para dividir
        self.rule = rule if rule else ""

        # Calcular los valores de Y en el nodo
        self.counts = Counter(Y)

        # Obtención de la impureza GINI basada en la distribución Y
        self.gini_impurity = self.get_GINI()

        # Ordenar los recuentos y guardar la predicción final del nodo
        counts_sorted = list(sorted(self.counts.items(), key=lambda item: item[1]))

        # Conseguir el último artículo
        yhat = None
        if len(counts_sorted) > 0:
            yhat = counts_sorted[-1][0]

        # Guardar en atributo de objeto. Este nodo predecirá la clase con la clase más frecuente
        self.yhat = yhat

        # Guardar el número de observaciones en el nodo
        self.n = len(Y)

        # Iniciar los nodos izquierdo y derecho como nodos vacíos
        self.left = None
        self.right = None

        # Valores predeterminados para divisiones
        self.best_feature = None
        self.best_value = None

    @staticmethod
    def GINI_impurity(y1_count: int, y2_count: int) -> float:
        """
        Dadas las observaciones de una clase binaria, calcule la impureza de GINI
        """
        # Asegurar los tipos correctos
        if y1_count is None:
            y1_count = 0

        if y2_count is None:
            y2_count = 0

        # Obteniendo las observaciones totales
        n = y1_count + y2_count

        # Si n es 0, devolvemos la menor impureza de gini posible
        if n == 0:
            return 0.0

        # Obtener la probabilidad de ver cada una de las clases.
        p1 = y1_count / n
        p2 = y2_count / n

        # Calculando GINI
        gini = 1 - (p1 ** 2 + p2 ** 2)

        # Devolviendo la impureza de gini
        return gini

    @staticmethod
    def ma(x: np.array, window: int) -> np.array:
        """
        Calcula el promedio móvil de la lista dada.
        """
        return np.convolve(x, np.ones(window), 'valid') / window

    def get_GINI(self):
        """
        Función para calcular la impureza GINI de un nodo
        """
        # Obtener los recuentos de 0 y 1
        y1_count, y2_count = self.counts.get(0, 0), self.counts.get(1, 0)

        # Obteniendo la impureza GINI
        return self.GINI_impurity(y1_count, y2_count)

    def best_split(self) -> tuple:
        """
        Dadas las características X y los objetivos Y, calcula la mejor división
        para un árbol de decisiones
        """
        # Crear un conjunto de datos para dividir
        df = self.X.copy()
        df['Y'] = self.Y

        # Obteniendo la impureza GINI para la entrada base
        GINI_base = self.get_GINI()

        # Encontrar qué división produce la mejor ganancia de GINI
        max_gain = 0

        # Mejor función predeterminada y división
        best_feature = None
        best_value = None

        for feature in self.features:
            # Eliminando los valores perdidos
            Xdf = df.dropna().sort_values(feature)

            # Ordenando los valores y obteniendo el promedio móvil
            xmeans = self.ma(Xdf[feature].unique(), 2)

            for value in xmeans:
                # División del conjunto de datos
                left_counts = Counter(Xdf[Xdf[feature]<value]['Y'])
                right_counts = Counter(Xdf[Xdf[feature]>=value]['Y'])

                # Obtener la distribución Y de los diccionarios
                y0_left, y1_left, y0_right, y1_right = left_counts.get(0, 0), left_counts.get(1, 0), right_counts.get(0, 0), right_counts.get(1, 0)

                # Obteniendo las impurezas gini izquierda y derecha
                gini_left = self.GINI_impurity(y0_left, y1_left)
                gini_right = self.GINI_impurity(y0_right, y1_right)

                # Obtener el recuento de obs de las divisiones de datos izquierda y derecha
                n_left = y0_left + y1_left
                n_right = y0_right + y1_right

                # Calcular los pesos para cada uno de los nodos
                w_left = n_left / (n_left + n_right)
                w_right = n_right / (n_left + n_right)

                # Cálculo de la impureza ponderada de GINI
                wGINI = w_left * gini_left + w_right * gini_right

                # Calcular la ganancia de GINI
                GINIgain = GINI_base - wGINI

                # Comprobando si esta es la mejor división hasta ahora
                if GINIgain > max_gain:
                    best_feature = feature
                    best_value = value

                    # Estableciendo la mejor ganancia a la actual
                    max_gain = GINIgain

        return (best_feature, best_value)

    def grow_tree(self):
        """
        Método recursivo para crear el árbol de decisiones
        """
        # Haciendo un dataset a partir de los datos
        df = self.X.copy()
        df['Y'] = self.Y

        # Si hay GINI que ganar, dividimos más
        if (self.depth < self.max_depth) and (self.n >= self.min_samples_split):

            # Conseguir la mejor división
            best_feature, best_value = self.best_split()

            if best_feature is not None:
                # Guardar la mejor división en el nodo actual
                self.best_feature = best_feature
                self.best_value = best_value

                # Obteniendo los nodos izquierdo y derecho
                left_df, right_df = df[df[best_feature]<=best_value].copy(), df[df[best_feature]>best_value].copy()

                # Creando los nodos izquierdo y derecho
                left = Node(
                    left_df['Y'].values.tolist(),
                    left_df[self.features],
                    depth=self.depth + 1,
                    max_depth=self.max_depth,
                    min_samples_split=self.min_samples_split,
                    node_type='left_node',
                    rule=f"{best_feature} <= {round(best_value, 3)}"
                    )
                self.left = left
                self.left.grow_tree()

                right = Node(
                    right_df['Y'].values.tolist(),
                    right_df[self.features],
                    depth=self.depth + 1,
                    max_depth=self.max_depth,
                    min_samples_split=self.min_samples_split,
                    node_type='right_node',
                    rule=f"{best_feature} > {round(best_value, 3)}"
                    )

                self.right = right
                self.right.grow_tree()

    def print_info(self, width=4):
        """
        Método para imprimir la información sobre el árbol.
        """
        # Definiendo el número de espacios
        const = int(self.depth * width ** 1.5)
        spaces = "-" * const

        if self.node_type == 'root':
            print("Raiz")
        else:
            print(f"|{spaces} Regla de division: {self.rule}")
        print(f"{' ' * const}   | Impureza GINI del nodo: {round(self.gini_impurity, 2)}")
        print(f"{' ' * const}   | Distribución de clases en el nodo: {dict(self.counts)}")
        print(f"{' ' * const}   | Clase clasificada: {self.yhat}")

    def print_tree(self):
        """
        Imprime todo el árbol desde el nodo actual hasta la parte inferior
        """
        self.print_info()

        if self.left is not None:
            self.left.print_tree()

        if self.right is not None:
            self.right.print_tree()

    def predict(self, X:pd.DataFrame):
        """
        Método de predicción por lotes
        """
        predictions = []

        for _, x in X.iterrows():
            values = {}
            for feature in self.features:
                values.update({feature: x[feature]})

            predictions.append(self.predict_obs(values))

        return predictions

    def predict_obs(self, values: dict) -> int:
        """
        Método para clasificar la clase dada un conjunto de características
        """
        cur_node = self
        while cur_node.depth < cur_node.max_depth:
            # Atravesando los nodos hasta el final
            best_feature = cur_node.best_feature
            best_value = cur_node.best_value

            if cur_node.n < cur_node.min_samples_split:
                break

            if (values.get(best_feature) < best_value):
                if self.left is not None:
                    cur_node = cur_node.left
            else:
                if self.right is not None:
                    cur_node = cur_node.right

        return cur_node.yhat



In [ ]:
def to_graphviz(node):
    """
    Convierte el árbol de decisiones en un objeto Digraph de graphviz
    """
    graph = graphviz.Digraph()
    # Recorremos el árbol en preorden
    _to_graphviz(node, graph)
    return graph

def _to_graphviz(node, graph):
    """
    Método auxiliar que convierte un nodo en un subgrafo de graphviz
    """
    # Agregamos el nodo al grafo
    node_id = str(id(node))
    if node.node_type == "root":
                    #print(f"|{spaces} Regla de division: {self.rule}")
        #print(f"{' ' * const}   | Impureza GINI del nodo: {round(self.gini_impurity, 2)}")
        #print(f"{' ' * const}   | Distribución de clases en el nodo: {dict(self.counts)}")
        #print(f"{' ' * const}   | Clase clasificada: {self.yhat}")

        graph.node(node_id, label="GINI - {} \n clase - {}".format(round(node.gini_impurity,2),node.yhat), shape="rectangle")
    else:
        graph.node(node_id, label="Regla de division: {} \n GINI - {} \n clase - {}".format(node.rule,
                                                                             round(node.gini_impurity,2),node.yhat))

    # Si hay hijos, los agregamos recursivamente
    if node.left is not None:
        left_id = str(id(node.left))
        graph.edge(node_id, left_id, label="<= {}".format(node.best_value))
        _to_graphviz(node.left, graph)
    if node.right is not None:
        right_id = str(id(node.right))
        graph.edge(node_id, right_id, label="> {}".format(node.best_value))
        _to_graphviz(node.right, graph)

---
## 🔍 Fase 2 · Comprensión de los datos — datos del Titanic y construcción del árbol

Cargamos los datos del Titanic (`Age`, `Fare` → `Survived`) y de abandono de clientes (`telecom_churn.csv`).

📚 Lectura recomendada: [11]

In [ ]:

# Leer la información
d = pd.read_csv("sample_data/train.csv")[['Age', 'Fare', 'Survived']].dropna()

# Construyendo las matrices X e Y
X = d[['Age', 'Fare']]
Y = d['Survived'].values.tolist()

# Iniciar el nodo
root = Node(Y, X, max_depth=3, min_samples_split=100)

# Obtener la mejor división
root.grow_tree()

# Clasificación
Xsubset = X.copy()
Xsubset['yhat'] = root.predict(Xsubset)
print(Xsubset)

In [ ]:
graph = to_graphviz(root)

# Visualizar el árbol
graph.render('graph', format='png')
Image(filename='graph.png')



---



---
## 🤖 Fase 4 · Modelado — bosque aleatorio desde cero

Programamos `Node` y `RandomForestClassifier` desde cero y visualizamos el árbol con Graphviz.

## **Bosque Aleatorio**

Es un algoritmo de aprendizaje supervisado que, como ya se puede ver en su nombre, crea un bosque y lo hace de alguna manera aleatorio. Para decirlo en palabras simples: el Bosque Aleatorio crea múltiples árboles de decisión y los combina para obtener una predicción más precisa y estable.

![Bagging: muestras bootstrap, clasificadores y votación](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch07/images/07_06.png)
*Figura: Bagging: muestras bootstrap, clasificadores y votación. [10]*

![Regiones de decisión de un bosque aleatorio sobre Iris](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_22.png)
*Figura: Regiones de decisión de un bosque aleatorio sobre Iris. [10]*

#### **Modelado bosque aleatorio desde 0**

In [ ]:
"""
Código que alberga la clase que crea y usa el clasificador de bosque aleatorio
"""

import pandas as pd
import numpy as np
import random
from collections import Counter
from tqdm import tqdm


from sklearn.metrics import precision_score, recall_score

In [ ]:
class RandomForestTree():
    """
    Class that grows one random forest tree
    """
    def __init__(
        self,
        Y,
        X,
        min_samples_split=None,
        max_depth=None,
        depth=None,
        X_features_fraction=None,
        node_type=None,
        rule=None
    ):
        # Saving the data for the random forest
        self.Y = Y
        self.X = X

        # Saving the hyper parameters
        self.min_samples_split = min_samples_split if min_samples_split else 20
        self.max_depth = max_depth if max_depth else 5

        # Default current depth of the tree
        self.depth = depth if depth else 0

        # Saving the final feature list
        self.features = list(X.columns)

         # Type of node
        self.node_type = node_type if node_type else 'root'

        # Rule for spliting
        self.rule = rule if rule else ""

        # Calculating the counts of Y in the node
        self.counts = Counter(Y)

        # Getting the GINI impurity based on the Y distribution
        self.gini_impurity = self.get_GINI()

        # Getting the number of features
        self.n_features = len(self.features)

        # Saving the hyper parameters specific to the random forest
        self.X_features_fraction = X_features_fraction if X_features_fraction is not None else 1.0

        # Sorting the counts and saving the final prediction of the node
        counts_sorted = list(sorted(self.counts.items(), key=lambda item: item[1]))

        # Getting the last item
        yhat = None
        if len(counts_sorted) > 0:
            yhat = counts_sorted[-1][0]

        # Saving to object attribute. This node will predict the class with the most frequent class
        self.yhat = yhat

        # Saving the number of observations in the node
        self.n = len(Y)

        # Initiating the left and right nodes as empty nodes
        self.left = None
        self.right = None

        # Default values for splits
        self.best_feature = None
        self.best_value = None

    def get_random_X_colsample(self):
        # Getting the random subset of features
        n_ft = int(self.n_features * self.X_features_fraction)

        # Selecting random features without repetition
        features = random.sample(self.features, n_ft)

        # Subseting the X to chosen features
        X = self.X[features].copy()

        # Returning the subseted features
        return X

    @staticmethod
    def GINI_impurity(y1_count: int, y2_count: int) -> float:
        """
        Given the observations of a binary class calculate the GINI impurity
        """
        # Ensuring the correct types
        if y1_count is None:
            y1_count = 0

        if y2_count is None:
            y2_count = 0

        # Getting the total observations
        n = y1_count + y2_count

        # If n is 0 then we return the lowest possible gini impurity
        if n == 0:
            return 0.0

        # Getting the probability to see each of the classes
        p1 = y1_count / n
        p2 = y2_count / n

        # Calculating GINI
        gini = 1 - (p1 ** 2 + p2 ** 2)

        # Returning the gini impurity
        return gini

    @staticmethod
    def ma(x: np.array, window: int) -> np.array:
        """
        Calculates the moving average of the given list.
        """
        return np.convolve(x, np.ones(window), 'valid') / window

    def get_GINI(self):
        """
        Function to calculate the GINI impurity of a node
        """
        # Getting the 0 and 1 counts
        y1_count, y2_count = self.counts.get(0, 0), self.counts.get(1, 0)

        # Getting the GINI impurity
        return self.GINI_impurity(y1_count, y2_count)

    def best_split(self) -> tuple:
        """
        Given the X features and Y targets calculates the best split
        for a decision tree
        """
        # Creating a dataset for spliting
        df = self.X.copy()
        df['Y'] = self.Y

        # Getting the GINI impurity for the base input
        GINI_base = self.get_GINI()

        # Finding which split yields the best GINI gain
        max_gain = 0

        # Default best feature and split
        best_feature = None
        best_value = None

        # Getting a random subsample of features
        n_ft = int(self.n_features * self.X_features_fraction)

        # Selecting random features without repetition
        features_subsample = random.sample(self.features, n_ft)

        for feature in features_subsample:
            # Droping missing values
            Xdf = df.dropna().sort_values(feature)

            # Sorting the values and getting the rolling average
            xmeans = self.ma(Xdf[feature].unique(), 2)

            for value in xmeans:
                # Spliting the dataset
                left_counts = Counter(Xdf[Xdf[feature]<value]['Y'])
                right_counts = Counter(Xdf[Xdf[feature]>=value]['Y'])

                # Getting the Y distribution from the dicts
                y0_left, y1_left, y0_right, y1_right = left_counts.get(0, 0), left_counts.get(1, 0), right_counts.get(0, 0), right_counts.get(1, 0)

                # Getting the left and right gini impurities
                gini_left = self.GINI_impurity(y0_left, y1_left)
                gini_right = self.GINI_impurity(y0_right, y1_right)

                # Getting the obs count from the left and the right data splits
                n_left = y0_left + y1_left
                n_right = y0_right + y1_right

                # Calculating the weights for each of the nodes
                w_left = n_left / (n_left + n_right)
                w_right = n_right / (n_left + n_right)

                # Calculating the weighted GINI impurity
                wGINI = w_left * gini_left + w_right * gini_right

                # Calculating the GINI gain
                GINIgain = GINI_base - wGINI

                # Checking if this is the best split so far
                if GINIgain > max_gain:
                    best_feature = feature
                    best_value = value

                    # Setting the best gain to the current one
                    max_gain = GINIgain

        return (best_feature, best_value)

    def grow_tree(self):
        """
        Recursive method to create the decision tree
        """
        # If there is GINI to be gained, we split further
        if (self.depth < self.max_depth) and (self.n >= self.min_samples_split):

            # Getting the best split
            best_feature, best_value = self.best_split()

            if best_feature is not None:
                # Saving the best split to the current node
                self.best_feature = best_feature
                self.best_value = best_value

                # Getting the left and right dataframe indexes
                left_index, right_index = self.X[self.X[best_feature]<=best_value].index, self.X[self.X[best_feature]>best_value].index

                # Extracting the left X and right X
                left_X, right_X = self.X[self.X.index.isin(left_index)], self.X[self.X.index.isin(right_index)]

                # Reseting the indexes
                left_X.reset_index(inplace=True, drop=True)
                right_X.reset_index(inplace=True, drop=True)

                # Extracting the left Y and the right Y
                left_Y, right_Y = [self.Y[x] for x in left_index], [self.Y[x] for x in right_index]

                # Creating the left and right nodes
                left = RandomForestTree(
                    left_Y,
                    left_X,
                    depth=self.depth + 1,
                    max_depth=self.max_depth,
                    min_samples_split=self.min_samples_split,
                    node_type='left_node',
                    rule=f"{best_feature} <= {round(best_value, 3)}"
                    )

                self.left = left
                self.left.grow_tree()

                right = RandomForestTree(
                    right_Y,
                    right_X,
                    depth=self.depth + 1,
                    max_depth=self.max_depth,
                    min_samples_split=self.min_samples_split,
                    node_type='right_node',
                    rule=f"{best_feature} > {round(best_value, 3)}"
                    )

                self.right = right
                self.right.grow_tree()

    def predict(self, X:pd.DataFrame):
        """
        Batch prediction method
        """
        predictions = []

        for _, x in X.iterrows():
            values = {}
            for feature in self.features:
                values.update({feature: x[feature]})

            predictions.append(self.predict_obs(values))

        return predictions

    def predict_obs(self, values: dict) -> int:
        """
        Method to predict the class given a set of features
        """
        cur_node = self
        while cur_node.depth < cur_node.max_depth:
            # Traversing the nodes all the way to the bottom
            best_feature = cur_node.best_feature
            best_value = cur_node.best_value

            if (cur_node.n < cur_node.min_samples_split) | (best_feature is None):
                break

            if (values.get(best_feature) < best_value):
                if self.left is not None:
                    cur_node = cur_node.left
            else:
                if self.right is not None:
                    cur_node = cur_node.right

        return cur_node.yhat

    def print_info(self, width=4):
        """
        Method to print the infromation about the tree
        """
        # Defining the number of spaces
        const = int(self.depth * width ** 1.5)
        spaces = "-" * const

        if self.node_type == 'root':
            print("Root")
        else:
            print(f"|{spaces} Split rule: {self.rule}")
        print(f"{' ' * const}   | GINI impurity of the node: {round(self.gini_impurity, 2)}")
        print(f"{' ' * const}   | Class distribution in the node: {dict(self.counts)}")
        print(f"{' ' * const}   | Predicted class: {self.yhat}")

    def print_tree(self):
        """
        Prints the whole tree from the current node to the bottom
        """
        self.print_info()

        if self.left is not None:
            self.left.print_tree()

        if self.right is not None:
            self.right.print_tree()

In [ ]:
class RandomForestClassifier():
    """
    Clase que crea un bosque aleatorio para problemas de clasificación
    """
    def __init__(
        self,
        Y: list,
        X: pd.DataFrame,
        min_samples_split=None,
        max_depth=None,
        n_trees=None,
        X_features_fraction=None,
        X_obs_fraction=None
    ):
        # Guardando los datos para el bosque aleatorio
        self.Y = Y
        self.X = X

        # Guardando los hiperparámetros
        self.min_samples_split = min_samples_split if min_samples_split else 20
        self.max_depth = max_depth if max_depth else 5

        # Guardar la lista de funciones final
        self.features = list(X.columns)

        # Obtener la cantidad de funciones
        self.n_features = len(self.features)

        # Guardar los hiperparámetros específicos del bosque aleatorio
        self.n_trees = n_trees if n_trees is not None else 30
        self.X_features_fraction = X_features_fraction if X_features_fraction is not None else 1.0
        self.X_obs_fraction = X_obs_fraction if X_obs_fraction is not None else 1.0

    def bootstrap_sample(self):
        """
        Función que crea una muestra de arranque con los parámetros de instancia de clase
        """
        # Muestreo del número de filas con repetición
        Xbootstrap = self.X.sample(frac=self.X_obs_fraction, replace=True)

        # Obtener el índice de muestras
        indexes = Xbootstrap.index

        # Obteniendo las correspondientes variables Y
        Ybootstrap = [self.Y[x] for x in indexes]

        # Descartando el índice de X
        Xbootstrap.reset_index(inplace=True, drop=True)

        # Devolviendo el par X, Y
        return Xbootstrap, Ybootstrap

    def grow_random_forest(self):
        """
        Método principal de la clase; Crea ** n_trees ** árboles aleatorios
        """
        # Lista de contener de árboles
        random_forest = []

        # Iterando
        for _ in tqdm(range(self.n_trees)):
            # Obteniendo la muestra bootstrapped
            X, Y = self.bootstrap_sample()

            # Iniciando el árbol aleatorio
            tree = RandomForestTree(
                Y=Y,
                X=X,
                min_samples_split=self.min_samples_split,
                max_depth=self.max_depth,
                X_features_fraction=self.X_features_fraction
                )

            # Agregar nodos nuevos al el arbol
            tree.grow_tree()

            # Agregar el árbol a la lista de árboles (el bosque)
            random_forest.append(tree)

        # Guardar la lista de bosques aleatorios en la memoria
        self.random_forest = random_forest

    def print_trees(self):
        """
        Método para imprimir todos los árboles generados en el clasificador
        """
        for i in range(self.n_trees):
            print("------ \n")
            print(f"Numero de arbol: {i + 1} \n")
            self.random_forest[i].print_tree()
            print("------ \n")

    def tree_predictions(self, X: pd.DataFrame) -> list:
        """
        Método para obtener las predicciones de todos los árboles.
        """
        predictions = []
        for i in range(self.n_trees):
            yhat = self.random_forest[i].predict(X)

            # Agregar al marcador de posición de predicción
            predictions.append(yhat)

        # Devolviendo la lista de predicciones
        return predictions

    def predict(self, X: pd.DataFrame) -> list:
        """
        Método para obtener la predicción final de todo el bosque aleatorio
        """
        # Obtener las predicciones de árboles individuales
        yhat = self.tree_predictions(X)

        # Guardar el número de obs en X
        n = X.shape[0]

        # Obtener el voto mayoritario de cada coordenada de la lista de predicción
        yhat_final = []

        for i in range(n):
            yhat_obs = [x[i] for x in yhat]

            # Obteniendo la entrada más frecuente
            counts = Counter(yhat_obs)
            most_common = counts.most_common(1)[0][0]

            # Agregar la entrada más común a la lista final de yhat
            yhat_final.append(most_common)

        # Devolviendo las predicciones finales
        return yhat_final



In [ ]:
# Lectura de datos para clasificación
d = pd.read_csv("sample_data/telecom_churn.csv")

# Configuración de las funciones utilizadas
features = [
    'AccountWeeks',
    'DataUsage',
    'DayMins',
    'DayCalls',
    'MonthlyCharge',
    'OverageFee',
    'RoamMins'
]

# Iniciando el objeto de bosque aleatorio
rf = RandomForestClassifier(
    Y=d['Churn'],
    X=d[features],
    min_samples_split=5,
    max_depth=4,
    n_trees=11,
    X_features_fraction=0.5
    )

# Creciendo el bosque aleatorio
rf.grow_random_forest()

# Imprimiendo los árboles
rf.print_trees()

# Haciendo predicciones
yhat = rf.predict(d[features])
d['yhat'] = yhat


---
## 📊 Fase 5 · Evaluación

Evaluamos con curva ROC/AUC, matriz de confusión, exactitud, sensibilidad y especificidad.

📚 Lectura recomendada: [6] [7]

**Evaluaciones de los modelos de clasificación**

### **Curva ROC**

Curva ROC (acrónimo de Receiver Operating Characteristic, o Característica Operativa del Receptor) es una representación gráfica de la sensibilidad frente a la especificidad para un sistema clasificador binario según se varía el umbral de discriminación.

![Curvas ROC con su AUC y la línea de adivinar al azar](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_10.png)
*Figura: Curvas ROC con su AUC y la línea de adivinar al azar. [10]*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# roc curve and auc score
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

In [ ]:
def plot_roc_curve(fpr, tpr):
    plt.plot(fpr, tpr, color="orange", label="ROC")
    plt.plot([0, 1], [0, 1], color="darkblue", linestyle="--")
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend()
    plt.show()

In [ ]:
data_X, class_label = make_classification(n_samples=1000, n_classes=2, weights=[1,1], random_state=1)

In [ ]:
trainX, testX, trainy, testy = train_test_split(data_X, class_label, test_size=0.3, random_state=1)

In [ ]:
model = RandomForestClassifier()
model.fit(trainX, trainy)

In [ ]:
probs = model.predict_proba(testX)

In [ ]:
probs

In [ ]:
classPredict = model.predict(testX)

In [ ]:
classPredict

In [ ]:
auc = roc_auc_score(testy, classPredict)
print('AUC: %.2f' % auc)

In [ ]:
fpr, tpr, thresholds = roc_curve(testy, classPredict)

In [ ]:
plot_roc_curve(fpr, tpr)

---
## 📊 Fase 5 · Evaluación — matriz de confusión y métricas

Evaluamos con curva ROC/AUC, matriz de confusión, exactitud, sensibilidad y especificidad.

### **Matriz de confusión**

Es una herramienta que permite la visualización del desempeño de un algoritmo que se emplea en aprendizaje supervisado. Cada columna de la matriz representa el número de predicciones de cada clase, mientras que cada fila representa a las instancias en la clase real. Uno de los beneficios de las matrices de confusión es que facilitan ver si el sistema está confundiendo dos clases.




**Errores**

$$ $$


 **Notas Importantes**

![Matriz de confusión: VP, FN, FP y VN](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_08.png)
*Figura: Matriz de confusión: VP, FN, FP y VN. [10]*

![Ejemplo de matriz de confusión con conteos reales](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_09.png)
*Figura: Ejemplo de matriz de confusión con conteos reales. [10]*

**Exactitud**: se refiere a lo cerca que está el resultado de una medición del valor verdadero. En términos estadísticos, la exactitud está relacionada con el sesgo de una estimación. También se conoce como Verdadero Positivo (o “True positive rate”).  Se representa como  la proporción de resultados verdaderos (tanto verdaderos positivos (VP) como verdaderos negativos (VN)) dividido entre el número total de casos

$$ $$

\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}
$$ $$

**Sensibilidad**: se conoce como Tasa de Verdaderos Positivos (True Positive Rate) ó TP. Es la proporción de casos positivos que fueron correctamente identificadas por el algoritmo

$$$$
\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

$$$$

**Especificidad**: También conocida como la Tasa de Verdaderos Negativos, (“true negative rate”) o TN. Se trata de los casos negativos que el algoritmo ha clasificado correctamente.  


\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}
$$ $$

**Ejemplo**

In [ ]:
from sklearn import datasets
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import plotly.graph_objs as go
import pandas as pd

In [ ]:
# Cargar datos
cancer = datasets.load_breast_cancer()

# create feature matrix
features = cancer.data

# create target vector
target = cancer.target

# create list of target class names
class_names = cancer.target_names

# split into training and test sets
# Crear conjunto de prueba conjunto de prueba
trainX, testX, trainy, testy = train_test_split(features, target, test_size=0.3, random_state=1)

model = RandomForestClassifier()
model.fit(trainX, trainy)

# entrenar modelo y hacer predicciones
target_predicted = model.predict(testX)

# crear matriz de confusión
matrix = confusion_matrix(testy, target_predicted)

# crear marco de datos de pandas Crear un conjunto de datos
dataframe = pd.DataFrame(matrix, index=class_names, columns=class_names)

# crear mapa de calor dibujar mapa de calor
sns.heatmap(dataframe, annot=True, cbar=None, cmap="Blues")
plt.title("Matrix de confusión"), plt.tight_layout()
plt.ylabel("Clases verdaderas"), plt.xlabel("Clase clasificada")
plt.show()

matrix

In [ ]:
# entrenar modelo y hacer predicciones
target_predicted = model.predict(testX)

In [ ]:
model.predict_proba(testX)

In [ ]:
model.predict(testX)

In [ ]:
auc = roc_auc_score(testy, target_predicted)
print('AUC: %.2f' % auc)

In [ ]:
import numpy as np
print("Exactitud:", np.trace(matrix) / matrix.sum())

In [ ]:
print("Sensibilidad (clase 1):", matrix[1, 1] / matrix[1].sum())

In [ ]:
print("Especificidad (clase 0):", matrix[0, 0] / matrix[0].sum())

In [ ]:
# Cargar datos
iris = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv')

print(iris)

# Convertir especies a valores numéricos
species_codes, species_names = pd.factorize(iris['species'])


# Definir traza
scatter_3d = go.Scatter3d(
    x=iris['sepal_length'],
    y=iris['sepal_width'],
    z=iris['petal_length'],
    mode='markers',
    marker=dict(
        size=5,
        color=species_codes,
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(title='Especies')
    ),
    text=iris['species']
)

# Definir diseño de gráfico
layout = go.Layout(
    title='Gráfico 3D del conjunto de datos Iris',
    scene=dict(
        xaxis=dict(title='Longitud del sépalo'),
        yaxis=dict(title='Ancho del sépalo'),
        zaxis=dict(title='Longitud del pétalo'),
    )
)

# Crear figura y traza
fig = go.Figure(data=[scatter_3d], layout=layout)

# Mostrar figura
fig.show()

# split into training and test sets
# Crear conjunto de prueba conjunto de prueba
trainX, testX, trainy, testy = train_test_split(iris[["sepal_length","sepal_width","petal_length","petal_width"]], iris[["species"]], test_size=0.3, random_state=1)

model = RandomForestClassifier()
model.fit(trainX, trainy)

# entrenar modelo y hacer predicciones
target_predicted = model.predict(testX)

# crear matriz de confusión
matrix = confusion_matrix(testy, target_predicted)

# crear marco de datos de pandas Crear un conjunto de datos
dataframe = pd.DataFrame(matrix, index=species_names, columns=species_names)

# crear mapa de calor dibujar mapa de calor
sns.heatmap(dataframe, annot=True, cbar=None, cmap="Blues")
plt.title("Matrix de confusión"), plt.tight_layout()
plt.ylabel("Clases verdaderas"), plt.xlabel("Clase clasificada")
plt.show()

In [ ]:
print(classification_report(testy, target_predicted))

---
## 🧪 Refuerzo: sobreajuste: la profundidad del árbol y por qué el bosque ayuda

Un árbol sin límite de profundidad memoriza el entrenamiento (**sobreajuste**): acierta casi el 100 % ahí, pero falla en datos nuevos. Un **bosque aleatorio** promedia muchos árboles entrenados con muestras *bootstrap* y subconjuntos de variables, lo que reduce la varianza [5]. Comparemos con `scikit-learn`:

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('sample_data/telecom_churn.csv')
feats = ['AccountWeeks', 'DataUsage', 'DayMins', 'DayCalls', 'MonthlyCharge', 'OverageFee', 'RoamMins']
X_tr, X_te, y_tr, y_te = train_test_split(df[feats], df['Churn'], test_size=0.3, random_state=1, stratify=df['Churn'])

prof = range(1, 16)
acc_tr, acc_te = [], []
for d in prof:
    t = DecisionTreeClassifier(max_depth=d, random_state=1).fit(X_tr, y_tr)
    acc_tr.append(t.score(X_tr, y_tr)); acc_te.append(t.score(X_te, y_te))

plt.plot(prof, acc_tr, 'o-', label='Entrenamiento')
plt.plot(prof, acc_te, 's-', label='Prueba')
plt.xlabel('Profundidad máxima del árbol'); plt.ylabel('Exactitud'); plt.legend(); plt.show()

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=1).fit(X_tr, y_tr)
print(f'Bosque aleatorio (prueba): {rf.score(X_te, y_te):.3f}  | mejor árbol individual: {max(acc_te):.3f}')
print(pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False).round(3))

### ✍️ Ejercicios
1. ¿Desde qué profundidad empieza a separarse la curva de entrenamiento de la de prueba? Esa es la señal del sobreajuste.
2. Cambie `n_estimators` a 10 y a 500: ¿cuánto cambia la exactitud de prueba?
3. Con la clase `Churn` desbalanceada, ¿por qué la **exactitud** puede ser engañosa? Calcule también *recall* de la clase 1.

---
## ✅ Conclusiones y siguientes pasos

Un árbol es fácil de **interpretar** pero inestable; el bosque sacrifica algo de interpretabilidad a cambio de **estabilidad y mejor generalización** [5]. Siempre se evalúa con datos no vistos y con métricas acordes al costo de cada error [7]. Lo siguiente es KNN, otro clasificador basado en ejemplos.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [8].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.

**[4]** Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning, 1*(1), 81–106. https://doi.org/10.1007/BF00116251

**[5]** Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

**[6]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[7]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[8]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[9]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[10]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[11]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a